<a href="https://colab.research.google.com/github/keshav123333/fastapi/blob/main/lecture_7_mlmodel_deploy/practiceinsurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("https://raw.githubusercontent.com/campusx-official/fastapi-demo-api/refs/heads/main/insurance.csv")

In [ ]:
df

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,67,119.8,1.56,2.92000,False,Jaipur,retired,High
1,36,101.1,1.83,34.28000,False,Chennai,freelancer,Low
2,39,56.8,1.64,36.64000,False,Indore,freelancer,Low
3,22,109.4,1.55,3.34000,True,Mumbai,student,Medium
4,69,62.2,1.60,3.94000,True,Indore,retired,High
...,...,...,...,...,...,...,...,...
95,36,52.8,1.57,19.64000,False,Indore,business_owner,Low
96,26,113.8,1.54,34.01000,False,Delhi,private_job,Low
97,52,60.8,1.80,44.86000,False,Hyderabad,freelancer,Low
98,27,101.1,1.82,28.30000,False,Kolkata,business_owner,Low


In [ ]:
df['age'].isna().sum()

np.int64(0)

In [ ]:
df["bmi"]=df["weight"]/(df["height"]**2)

In [ ]:
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    else:
        return "senior"
df["age_group"]=df["age"].apply(age_group)

In [ ]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]


# Feature 4: City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [ ]:
df["city_tier"]=df["city"].apply(city_tier)

In [ ]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"


df["lifestyle_risk"] = df.apply(lifestyle_risk, axis=1)

In [ ]:
df.drop(columns=['age', 'weight', 'height', 'smoker', 'city'],inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


In [ ]:
df

,income_lpa,occupation,insurance_premium_category,bmi,age_group,city_tier,lifestyle_risk
0,2.92000,retired,High,49.227482,senior,2,medium
1,34.28000,freelancer,Low,30.189017,adult,1,medium
2,36.64000,freelancer,Low,21.118382,adult,2,low
3,3.34000,student,Medium,45.535900,young,1,high
4,3.94000,retired,High,24.296875,senior,2,medium
...,...,...,...,...,...,...,...
95,19.64000,business_owner,Low,21.420747,adult,2,low
96,34.01000,private_job,Low,47.984483,adult,1,medium
97,44.86000,freelancer,Low,18.765432,middle_aged,1,low
98,28.30000,business_owner,Low,30.521676,adult,1,medium


In [ ]:

# Select features and target
X = df[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df["insurance_premium_category"]

In [ ]:
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [ ]:
col1=ColumnTransformer([
    ("ohe",OneHotEncoder(drop="first"),categorical_features)
],remainder="passthrough"
)

In [ ]:
pipe=Pipeline([
    ("preprocessing",col1),
    ("model",RandomForestClassifier())
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipe.fit(X_train, y_train)


/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe',
                                                  OneHotEncoder(drop='first'),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation',
                                                   'city_tier'])])),
                ('model', RandomForestClassifier())])

In [ ]:
y_pred = pipe.predict(X_test)
accuracy_score(y_test, y_pred)


0.8

In [ ]:
import pickle

with open("model.pkl","wb") as f:
  pickle.dump(pipe,f)